# FreightQuote AI — Member 1: RAG Knowledge Base Builder (Milestone 3, v4)
**Goal:** scrape 150+ unique websites/PDFs on Incoterms, customs, HS codes, shipping documents, demurrage, detention, trade finance, **freight rates, surcharges, ports, congestion, weather, route risks and marine insurance**; de-duplicate them; save PDFs to `MyDrive/FreightQuote_AI/rag_pdfs/`; build chunks, an FTS5 index and a FAISS vector index; test the RAG pipeline with numbered citations; record statistics.

**v2 additions (kept):** `FREIGHT_SEEDS` (37 extra freight sources) · multi-round auto-expansion to 150+ · URL-normalization + SHA-256 content-hash dedup · FAISS index.

**v3 additions**
6. **Complete topic coverage** — 7 new topics (`freight_rates`, `surcharges`, `ports`, `congestion`, `weather`, `route_risks`, `marine_insurance`) with Wikipedia + official/industry sources and a few PDFs. Coverage is now checked **per topic** (min. documents, min. chunks, key terms *inside that topic*).
7. **12 RAG test questions** (was 4) covering every topic.
8. **Numbered citations** — answers are built from retrieved sentences, each tagged `[1]`, `[2]`…, with a Sources list; a verifier proves every `[n]` maps to a real source and that every cited sentence appears verbatim in the cited chunk.
9. **Statistics recorded** — `kb_statistics.json` / `kb_statistics.md`: unique documents, website/PDF counts, duplicates removed, chunks, per-topic counts, FTS5 + FAISS index stats, coverage and RAG-test results.

**v4 additions**
10. **BM25 index** — `rank-bm25` is installed, a `BM25Okapi` index is built from the generated chunks and saved with metadata (`bm25_index.pkl`, `bm25_meta.json`, `bm25_tokenizer.py`). BM25 is now the third retriever in the hybrid RRF fusion (FAISS + FTS5 + BM25).
11. **Self-check & auto-repair** — before the Verification cell, every topic is tested; any `LOW` topic automatically gets extra sources and all indexes (FTS5, FAISS, BM25) are rebuilt (up to 3 attempts).
12. **RAG questions retry** — a question that fails is retried with wider retrieval; the summary prints how many of the 12 passed and whether all citations are valid.
13. **Final checklist cell** — prints PASS/FAIL for: 150+ unique documents, FAISS, BM25, all 13 topics, 12/12 RAG questions, valid citations. Saved to `final_checklist.json`.

14. **Execution gate (cell 3b)** — 150+ unique sources are *required*: the notebook auto-fetches a reserve list and widens the crawl if short, writes `source_count_proof.json`, and **raises an error** (stops “Run all”) if still below 150. The final checklist also asserts every requirement.

**Run order:** run all cells top to bottom → check the *Verification* cell prints `PASS`, the *RAG test* cell prints `PASS`, and the statistics cell writes `kb_statistics.md` → load `rag_data/` in the final application notebook.

In [ ]:
!pip -q install requests beautifulsoup4 lxml pypdf sentence-transformers faiss-cpu numpy rank-bm25

In [ ]:
import os, re, json, time, sqlite3, hashlib, csv
from urllib.parse import urljoin, urlparse, urlunparse, parse_qsl, urlencode
from collections import Counter, defaultdict
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from google.colab import drive
drive.mount('/content/drive')

BASE   = '/content/drive/MyDrive/FreightQuote_AI'
PDF_DIR  = f'{BASE}/rag_pdfs'      # required location
DATA_DIR = f'{BASE}/rag_data'      # chunks + dbs + faiss for the final app
for d in (PDF_DIR, DATA_DIR): os.makedirs(d, exist_ok=True)

TARGET_SOURCES = 150               # minimum UNIQUE successfully processed sources
CHUNK_CHARS, OVERLAP = 900, 150
EMB_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; FreightQuoteAI-RAG-Builder/2.0; student project)'}
print('Drive ready:', BASE)

## 0. De-duplication helpers
* `normalize_url` — lowercases host, drops `www.`, forces https, removes fragments, tracking params (`utm_*`, `gclid`, `fbclid`…), default ports and trailing slashes, and sorts query params.
* `content_hash` — SHA-256 of whitespace/case-normalized text, so the same page served from two URLs is only kept once.

In [ ]:
TRACKING = {'gclid','fbclid','msclkid','mc_cid','mc_eid','ref','ref_src','source','sessionid','sid'}

def normalize_url(u):
    u = u.strip()
    p = urlparse(u)
    scheme = 'https'
    host = (p.hostname or '').lower()
    if host.startswith('www.'): host = host[4:]
    port = p.port
    netloc = host if (port in (None, 80, 443)) else f'{host}:{port}'
    path = re.sub(r'/{2,}', '/', p.path or '/')
    if len(path) > 1 and path.endswith('/'): path = path[:-1]
    if path.lower().endswith(('/index.html', '/index.htm', '/index.php')):
        path = path.rsplit('/', 1)[0] or '/'
    q = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=False)
         if not k.lower().startswith('utm_') and k.lower() not in TRACKING]
    return urlunparse((scheme, netloc, path, '', urlencode(sorted(q)), ''))

def norm_text(t):
    return re.sub(r'\s+', ' ', t.lower()).strip()

def content_hash(t):
    return hashlib.sha256(norm_text(t).encode('utf-8')).hexdigest()

# quick self-test
assert normalize_url('http://www.Example.com/a/?utm_source=x#top') == normalize_url('https://example.com/a')
print('normalize_url OK')

## 1. Source list
Topics are tagged so we can check coverage. `wiki_topics` + `OFFICIAL` are the original seeds; **`FREIGHT_SEEDS`** and **`wiki_freight_extra`** are the new additions. You can still add your own in `EXTRA_SOURCES` or upload a CSV (`url,topic`).

In [ ]:
WIKI = "https://en.wikipedia.org/wiki/"
wiki_topics = {
 "incoterms": ["Incoterms","Free_on_board","Cost,_insurance_and_freight","Ex_Works","Delivered_duty_paid","Free_carrier","Delivered_at_place","Carriage_and_Insurance_Paid_To","Carriage_paid_to","Free_alongside_ship","Cost_and_freight","Delivered_at_terminal","Delivered_duty_unpaid","Delivered_at_frontier","Delivered_ex_ship","Incoterms_2020"],
 "customs": ["Customs","Customs_broker","Customs_declaration","Tariff","Duty-free_shop","Import_quota","Rules_of_origin","Free_trade_zone","Bonded_warehouse","Customs_union","World_Customs_Organization","Authorized_economic_operator","Anti-dumping_duty","Countervailing_duties","Trade_barrier","Embargo","Export_control","De_minimis","Customs_valuation","Customs_clearance","Drawback_(customs)","Temporary_import","ATA_Carnet","Single_Window_(trade)","Automated_Commercial_Environment","Container_Security_Initiative","Customs-Trade_Partnership_Against_Terrorism","Advance_ruling","Trade_facilitation","WTO_Trade_Facilitation_Agreement","General_Agreement_on_Tariffs_and_Trade","World_Trade_Organization","Generalized_System_of_Preferences","Free-trade_agreement","Preferential_trade","Most_favoured_nation","Non-tariff_barriers_to_trade","Sanitary_and_phytosanitary_measures"],
 "hs_codes": ["Harmonized_System","Harmonized_Tariff_Schedule_of_the_United_States","Combined_Nomenclature","TARIC","Standard_International_Trade_Classification","Customs_tariff","Schedule_B","Tariff_classification"],
 "shipping_documents": ["Bill_of_lading","Commercial_invoice","Packing_list","Certificate_of_origin","Sea_waybill","Air_waybill","Manifest_(cargo)","Shipping_container","Delivery_order","Letter_of_indemnity","Mate%27s_receipt","Cargo_insurance","Marine_insurance","Phytosanitary_certificate","Dangerous_goods","Bill_of_entry","Shipping_agency","Freight_forwarder","Freight_transport","Intermodal_container","Containerization","Less_than_container_load","Full_container_load","Twenty-foot_equivalent_unit","Break_bulk_cargo","Bulk_cargo","Roll-on/roll-off","Cargo","Freight_rate","Freight_Rate_Index","Container_ship","Port","Terminal_operating_system","Drayage","Cargo_ship","Tanker_(ship)","Reefer_ship","Intermodal_freight_transport","Multimodal_transport","Third-party_logistics","Supply_chain","Logistics","Bunker_fuel","Bunker_adjustment_factor","Currency_adjustment_factor","Fuel_surcharge","Peak_season_surcharge","Terminal_handling_charge","Cargo_volume_weight","Dimensional_weight","Shipping_line","Ocean_freight","Air_cargo","Rail_freight_transport","Trucking_industry","Last_mile_(transportation)","Cross-docking"],
 "demurrage_detention": ["Demurrage","Detention_(shipping)","Laytime","Despatch_money","Charter_party","Time_charter","Voyage_charter","Notice_of_readiness","Port_congestion","Free_time_(shipping)","Per_diem_(shipping)","Force_majeure","Hague-Visby_Rules","Hamburg_Rules","Rotterdam_Rules","Carriage_of_Goods_by_Sea_Act","General_average","York-Antwerp_Rules","Maritime_law","Admiralty_law","Suez_Canal","Panama_Canal","Strait_of_Hormuz","Strait_of_Malacca","Bab-el-Mandeb","Red_Sea_crisis","2021_Suez_Canal_obstruction","Global_supply_chain_crisis"],
 "trade_finance": ["Letter_of_credit","Documentary_collection","Trade_finance","Bank_guarantee","Open_account","Cash_in_advance","Supply_chain_finance","Factoring_(finance)","Forfaiting","Export_credit_agency","Uniform_Customs_and_Practice_for_Documentary_Credits","Standby_letter_of_credit","Export_credit","Foreign_exchange_risk","Hedge_(finance)","Forward_contract","Currency_risk","Payment_terms","Incoterms_and_payment","Documents_against_payment","Bankers%27_acceptance","Credit_insurance","Export_finance","International_Chamber_of_Commerce"],
}
# NEW: extra freight/logistics Wikipedia pages (backup volume; duplicates are removed automatically)
wiki_freight_extra = {
 "shipping_documents": ["Freight_broker","Less-than-truckload_shipping","Cold_chain","Transshipment","Feeder_ship","Maritime_transport","Port_of_Rotterdam","Port_of_Singapore","Port_of_Shanghai","Jawaharlal_Nehru_Port","Container_port","Air_freight","Hub_and_spoke","Freight_train","Inland_port","Warehouse","Bulk_carrier","Cargo_airline"],
 "customs": ["Customs_Act","Import_duty","Export_duty","Value-added_tax","Goods_and_Services_Tax_(India)","Trade_agreement","Trade_war","Tariffs_in_the_first_Trump_administration"],
}
sources = []
for topic, titles in list(wiki_topics.items()) + list(wiki_freight_extra.items()):
    for t in titles: sources.append({"url": WIKI+t, "topic": topic, "kind": "html"})

OFFICIAL = [
 ("https://iccwbo.org/business-solutions/incoterms-rules/", "incoterms"),
 ("https://iccwbo.org/business-solutions/incoterms-rules/incoterms-2020/", "incoterms"),
 ("https://www.trade.gov/know-your-incoterms", "incoterms"),
 ("https://www.trade.gov/export-solutions", "customs"),
 ("https://www.trade.gov/documentation-requirements", "shipping_documents"),
 ("https://www.trade.gov/trade-financing", "trade_finance"),
 ("https://www.trade.gov/methods-payment", "trade_finance"),
 ("https://www.trade.gov/export-documents", "shipping_documents"),
 ("https://www.cbp.gov/trade/basic-import-export", "customs"),
 ("https://www.cbp.gov/trade/programs-administration/entry-summary", "customs"),
 ("https://www.cbp.gov/trade/rulings", "hs_codes"),
 ("https://www.cbp.gov/trade/priority-issues/classification", "hs_codes"),
 ("https://www.cbp.gov/trade/programs-administration/importing-exporting-process", "customs"),
 ("https://www.cbp.gov/trade/basic-import-export/importer-exporter-tools", "customs"),
 ("https://www.wcoomd.org/en/topics/nomenclature/overview/what-is-the-harmonized-system.aspx", "hs_codes"),
 ("https://www.wcoomd.org/en/topics/nomenclature/overview.aspx", "hs_codes"),
 ("https://www.wto.org/english/tratop_e/tradfa_e/tradfa_e.htm", "customs"),
 ("https://www.wto.org/english/tratop_e/cusval_e/cusval_e.htm", "customs"),
 ("https://www.wto.org/english/tratop_e/roi_e/roi_e.htm", "customs"),
 ("https://www.cbic.gov.in/", "customs"),
 ("https://www.dgft.gov.in/CP/", "customs"),
 ("https://www.indiantradeportal.in/", "hs_codes"),
 ("https://www.icegate.gov.in/", "customs"),
 ("https://unctad.org/topic/transport-and-trade-logistics", "demurrage_detention"),
 ("https://unctad.org/topic/transport-and-trade-logistics/review-of-maritime-transport", "demurrage_detention"),
 ("https://www.fmc.gov/resources-services/demurrage-and-detention/", "demurrage_detention"),
 ("https://www.fmc.gov/rules-regulations/", "demurrage_detention"),
 ("https://www.imo.org/en/About/Pages/Default.aspx", "shipping_documents"),
 ("https://www.ifc.org/en/what-we-do/sector-expertise/financial-institutions/trade-finance", "trade_finance"),
 ("https://www.adb.org/what-we-do/trade-finance-program", "trade_finance"),
 ("https://www.exim.gov/", "trade_finance"),
 ("https://iccwbo.org/business-solutions/trade-finance/", "trade_finance"),
 ("https://www.ups.com/us/en/supplychain/resources/knowledge-center.page", "shipping_documents"),
 ("https://www.dhl.com/global-en/home/logistics-glossary.html", "shipping_documents"),
 ("https://www.maersk.com/support/faq", "demurrage_detention"),
 ("https://www.maersk.com/local-information", "demurrage_detention"),
 ("https://www.msc.com/en/help/faq", "demurrage_detention"),
]
for u,t in OFFICIAL: sources.append({"url":u,"topic":t,"kind":"html"})

# ===================== NEW: FREIGHT_SEEDS (30+ reliable freight & logistics sources) =====================
FREIGHT_SEEDS = [
 # carriers / shipping lines
 ("https://www.maersk.com/logistics-explained", "shipping_documents"),
 ("https://www.maersk.com/support/faq/detention-and-demurrage", "demurrage_detention"),
 ("https://www.hapag-lloyd.com/en/online-business/book/demurrage-and-detention.html", "demurrage_detention"),
 ("https://www.cma-cgm.com/local/india", "shipping_documents"),
 ("https://www.one-line.com/en/standard-page/demurrage-detention", "demurrage_detention"),
 ("https://www.evergreen-line.com/static/jsp/FAQ.jsp", "shipping_documents"),
 # express / forwarders / 3PL
 ("https://www.fedex.com/en-us/shipping/international.html", "shipping_documents"),
 ("https://www.fedex.com/en-us/customs/international-shipping.html", "customs"),
 ("https://www.dhl.com/global-en/home/our-divisions/global-forwarding.html", "shipping_documents"),
 ("https://www.dhl.com/global-en/home/customer-service/glossary.html", "shipping_documents"),
 ("https://www.dbschenker.com/global/about/glossary", "shipping_documents"),
 ("https://www.kuehne-nagel.com/en-gb/knowledge", "shipping_documents"),
 ("https://www.flexport.com/glossary/", "shipping_documents"),
 ("https://www.flexport.com/blog/", "shipping_documents"),
 ("https://www.freightos.com/freight-resources/", "shipping_documents"),
 ("https://www.searates.com/reference/", "shipping_documents"),
 ("https://www.searates.com/blog/", "shipping_documents"),
 ("https://www.freightwaves.com/news", "shipping_documents"),
 # industry bodies / regulators
 ("https://www.iata.org/en/programs/cargo/", "shipping_documents"),
 ("https://www.iata.org/en/programs/cargo/dgr/", "shipping_documents"),
 ("https://www.iata.org/en/programs/cargo/e/eawb/", "shipping_documents"),
 ("https://www.ics-shipping.org/", "shipping_documents"),
 ("https://www.worldshipping.org/", "shipping_documents"),
 ("https://www.bimco.org/", "demurrage_detention"),
 ("https://fiata.org/", "shipping_documents"),
 ("https://www.fmcsa.dot.gov/", "shipping_documents"),
 ("https://www.bts.gov/topics/freight-transportation", "shipping_documents"),
 ("https://www.transportation.gov/freight", "shipping_documents"),
 # government / multilateral customs & trade
 ("https://taxation-customs.ec.europa.eu/customs_en", "customs"),
 ("https://taxation-customs.ec.europa.eu/customs-4/calculation-customs-duties/customs-valuation_en", "customs"),
 ("https://www.gov.uk/topic/business-tax/import-export", "customs"),
 ("https://www.gov.uk/guidance/classifying-your-goods-for-import-and-export", "hs_codes"),
 ("https://www.intracen.org/", "customs"),
 ("https://www.worldbank.org/en/topic/trade", "customs"),
 ("https://www.worldbank.org/en/topic/transport", "shipping_documents"),
 ("https://www.hts.usitc.gov/", "hs_codes"),
 # reference explainers (stable URLs)
 ("https://www.investopedia.com/terms/i/incoterms.asp", "incoterms"),
 ("https://www.investopedia.com/terms/f/fob.asp", "incoterms"),
 ("https://www.investopedia.com/terms/l/letterofcredit.asp", "trade_finance"),
 ("https://www.investopedia.com/terms/b/billoflading.asp", "shipping_documents"),
]
for u,t in FREIGHT_SEEDS: sources.append({"url":u,"topic":t,"kind":"html"})
print("FREIGHT_SEEDS added:", len(FREIGHT_SEEDS))

# ===================== NEW (v3): COMPLETE TOPIC COVERAGE =====================
# rates, surcharges, ports, congestion, weather, route risks, marine insurance
COVERAGE_WIKI = {
 "freight_rates": ["Baltic_Dry_Index","Baltic_Exchange","Freightos_Baltic_Index","Drewry_Shipping_Consultants","Spot_market","Ocean_Shipping_Reform_Act_of_2022","Shipbroking"],
 "surcharges": ["IMO_2020","Emission_control_area","EU_Emissions_Trading_System","MARPOL_73/78","Surcharge","International_Maritime_Organization","Carbon_tax"],
 "ports": ["Port_of_Los_Angeles","Port_of_Long_Beach","Port_of_Antwerp-Bruges","Port_of_Hamburg","Port_of_Busan","Port_of_Ningbo-Zhoushan","Port_of_Jebel_Ali","Mundra_Port","Chennai_Port","Visakhapatnam_Port","List_of_busiest_container_ports","Deepwater_port","Sagarmala","Port_authority","Seaport"],
 "congestion": ["Bullwhip_effect","Queueing_theory","Lead_time","Just-in-time_manufacturing","Supply_chain_management","Supply_chain_resilience"],
 "weather": ["Tropical_cyclone","Typhoon","Hurricane","Monsoon","Storm_surge","Weather_routing","Beaufort_scale","Rogue_wave","Fog","Icebreaker","Drought","El_Ni%C3%B1o%E2%80%93Southern_Oscillation"],
 "route_risks": ["Piracy_in_the_Gulf_of_Aden","Piracy","Cape_of_Good_Hope","Maritime_security","Taiwan_Strait","Northern_Sea_Route","Strait_of_Gibraltar","Turkish_Straits","Economic_sanctions","Blockade"],
 "marine_insurance": ["Marine_Insurance_Act_1906","Lloyd%27s_of_London","Protection_and_indemnity_insurance","Institute_Cargo_Clauses","Insurance","Reinsurance","Underwriting","Insurable_interest","Constructive_total_loss","Loss_adjuster"],
}
COVERAGE_OFFICIAL = [
 # freight rates
 ("https://www.drewry.co.uk/supply-chain-advisors/supply-chain-expertise/world-container-index-assessed-by-drewry", "freight_rates"),
 ("https://www.balticexchange.com/en/index.html", "freight_rates"),
 ("https://www.fmc.gov/resources-services/", "freight_rates"),
 # surcharges
 ("https://www.ups.com/us/en/support/shipping-support/shipping-costs-rates/fuel-surcharges.page", "surcharges"),
 ("https://www.fedex.com/en-us/shipping/surcharges.html", "surcharges"),
 ("https://www.dhl.com/global-en/home/customer-service/fuel-surcharge.html", "surcharges"),
 ("https://www.imo.org/en/MediaCentre/HotTopics/Pages/Sulphur-2020.aspx", "surcharges"),
 ("https://www.hapag-lloyd.com/en/online-business/book/surcharges.html", "surcharges"),
 # ports
 ("https://www.portofrotterdam.com/en", "ports"),
 ("https://www.portoflosangeles.org/", "ports"),
 ("https://www.polb.com/", "ports"),
 ("https://www.portofantwerpbruges.com/en", "ports"),
 ("https://www.singaporepsa.com/", "ports"),
 ("https://www.jnport.gov.in/", "ports"),
 ("https://www.aapaports.org/", "ports"),
 ("https://www.marad.dot.gov/ports/", "ports"),
 ("https://www.sagarmala.gov.in/", "ports"),
 # congestion
 ("https://www.newyorkfed.org/research/policy/gscpi", "congestion"),
 ("https://www.portoflosangeles.org/business/supply-chain/port-optimizer", "congestion"),
 # weather
 ("https://www.nhc.noaa.gov/", "weather"),
 ("https://www.weather.gov/marine/", "weather"),
 ("https://public.wmo.int/en", "weather"),
 ("https://mausam.imd.gov.in/", "weather"),
 ("https://www.metoffice.gov.uk/weather/learn-about/weather", "weather"),
 ("https://oceanservice.noaa.gov/facts/", "weather"),
 # route risks
 ("https://www.ukmto.org/", "route_risks"),
 ("https://www.icc-ccs.org/piracy-reporting-centre", "route_risks"),
 ("https://www.marad.dot.gov/msci/", "route_risks"),
 ("https://www.suezcanal.gov.eg/English/Pages/default.aspx", "route_risks"),
 ("https://pancanal.com/en/", "route_risks"),
 ("https://www.bimco.org/news-and-trends/", "route_risks"),
 # marine insurance
 ("https://www.lloyds.com/", "marine_insurance"),
 ("https://iumi.com/", "marine_insurance"),
 ("https://www.igpandi.org/", "marine_insurance"),
 ("https://www.investopedia.com/terms/m/marine-insurance.asp", "marine_insurance"),
]
# PDFs (saved to rag_pdfs/). UNCTAD Review of Maritime Transport covers rates, ports, congestion and route risks.
COVERAGE_PDFS = [
 ("https://unctad.org/system/files/official-document/rmt2023_en.pdf", "ports"),
 ("https://unctad.org/system/files/official-document/rmt2022_en.pdf", "congestion"),
 ("https://unctad.org/system/files/official-document/rmt2021_en_0.pdf", "freight_rates"),
 ("https://unctad.org/system/files/official-document/rmt2024_en.pdf", "route_risks"),
]
for topic, titles in COVERAGE_WIKI.items():
    for t in titles: sources.append({"url": WIKI+t, "topic": topic, "kind": "html"})
for u,t in COVERAGE_OFFICIAL: sources.append({"url":u,"topic":t,"kind":"html"})
for u,t in COVERAGE_PDFS: sources.append({"url":u,"topic":t,"kind":"pdf"})
print("Coverage seeds added:", sum(len(v) for v in COVERAGE_WIKI.values()), "wiki +", len(COVERAGE_OFFICIAL), "official +", len(COVERAGE_PDFS), "pdf")

# existing pages that belong to a new topic are RE-TAGGED (kept once, no duplicate document)
COVERAGE_RETAG = {
 "Fuel_surcharge":"surcharges","Bunker_adjustment_factor":"surcharges","Currency_adjustment_factor":"surcharges",
 "Peak_season_surcharge":"surcharges","Terminal_handling_charge":"surcharges",
 "Freight_rate":"freight_rates","Freight_Rate_Index":"freight_rates",
 "Port":"ports","Port_of_Rotterdam":"ports","Port_of_Singapore":"ports","Port_of_Shanghai":"ports",
 "Jawaharlal_Nehru_Port":"ports","Container_port":"ports","Inland_port":"ports",
 "Port_congestion":"congestion","Global_supply_chain_crisis":"congestion",
 "Marine_insurance":"marine_insurance","Cargo_insurance":"marine_insurance","General_average":"marine_insurance",
 "Red_Sea_crisis":"route_risks","Suez_Canal":"route_risks","Panama_Canal":"route_risks","Strait_of_Hormuz":"route_risks",
 "Strait_of_Malacca":"route_risks","Bab-el-Mandeb":"route_risks","2021_Suez_Canal_obstruction":"route_risks",
}

# ---- add your own below (PDF links end with .pdf; they are saved to rag_pdfs/) ----
EXTRA_SOURCES = [
 # {"url": "https://example.org/guide.pdf", "topic": "hs_codes"},
]
for s in EXTRA_SOURCES: sources.append({**s, "kind": "pdf" if s["url"].lower().endswith(".pdf") else "html"})

# ---- optional: upload sources.csv with columns url,topic ----
try:
    from google.colab import files
    print("Optional: upload sources.csv (url,topic) or press Cancel.")
    up = files.upload()
    for name, data in up.items():
        for r in csv.DictReader(data.decode().splitlines()):
            u = r["url"].strip()
            sources.append({"url": u, "topic": r["topic"].strip(), "kind": "pdf" if u.lower().endswith(".pdf") else "html"})
except Exception as e:
    print("No upload:", e)

# ---- URL-normalization dedup of the seed list ----
seen_seed = set(); uniq = []
dup_log = []   # (type, url, duplicate_of)
for s in sources:
    n = normalize_url(s["url"])
    if n in seen_seed:
        dup_log.append(("seed_url", s["url"], n)); continue
    seen_seed.add(n); s["norm_url"] = n; uniq.append(s)
sources = uniq
_retag = {normalize_url(WIKI+k): v for k, v in COVERAGE_RETAG.items()}
for s in sources:
    if s["norm_url"] in _retag: s["topic"] = _retag[s["norm_url"]]
print(len(sources), "unique seed sources (", len(dup_log), "duplicate seed URLs removed );", dict(Counter(s["topic"] for s in sources)))

## 2. Fetch & extract text (with URL + content de-duplication)
HTML pages are cleaned (scripts/nav removed). PDFs are saved to Drive `rag_pdfs/` and text-extracted. A source is **skipped** if its normalized URL (including the post-redirect URL) or its content hash has already been processed. Failed sources are logged, never crash the run.

In [ ]:
def clean_html(html):
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script","style","nav","footer","header","aside","form","noscript","svg"]): t.decompose()
    for t in soup.select(".navbox,.mw-editsection,.reference,.hatnote,#toc,.infobox,.sidebar"): t.decompose()
    main = soup.find("main") or soup.find(id="mw-content-text") or soup.find("article") or soup.body or soup
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = main.get_text("\n")
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text).strip()
    return title, text, soup

def pdf_text(path):
    r = PdfReader(path); out=[]
    for p in r.pages:
        try: out.append(p.extract_text() or "")
        except Exception: pass
    return re.sub(r"\s+\n", "\n", "\n".join(out)).strip()

def fetch(src):
    url = src["url"]
    r = requests.get(url, headers=HEADERS, timeout=30, allow_redirects=True)
    r.raise_for_status()
    final_url = normalize_url(r.url)
    ctype = r.headers.get("content-type","").lower()
    if url.lower().endswith(".pdf") or "pdf" in ctype:
        fn = re.sub(r"[^A-Za-z0-9._-]+","_", urlparse(url).path.split("/")[-1] or hashlib.md5(url.encode()).hexdigest())
        if not fn.lower().endswith(".pdf"): fn += ".pdf"
        path = os.path.join(PDF_DIR, fn)
        with open(path,"wb") as fh: fh.write(r.content)
        return {"title": fn, "text": pdf_text(path)[:250000], "type":"pdf", "file": fn, "links": [], "final_url": final_url}
    title, text, soup = clean_html(r.text)
    links = [urljoin(r.url, a["href"]).split("#")[0] for a in soup.find_all("a", href=True)]
    return {"title": title, "text": text, "type":"html", "file": "", "links": links, "final_url": final_url}

docs, failed = [], []
seen_urls   = set()          # normalized URLs already processed (requested + final)
seen_hashes = {}             # content hash -> normalized url that owns it

def process(src_list, verbose=True):
    for s in src_list:
        n = s.get("norm_url") or normalize_url(s["url"])
        s["norm_url"] = n
        if n in seen_urls:
            dup_log.append(("url", s["url"], n)); continue
        seen_urls.add(n)
        try:
            d = fetch(s)
            if len(d["text"]) < 400: raise ValueError("too little text")
            if d["final_url"] in seen_urls and d["final_url"] != n:
                dup_log.append(("redirect_url", s["url"], d["final_url"])); continue
            seen_urls.add(d["final_url"])
            h = content_hash(d["text"])
            if h in seen_hashes:
                dup_log.append(("content_hash", s["url"], seen_hashes[h]));
                if verbose: print(f"[dup] {s['url'][:80]} == {seen_hashes[h][:60]}")
                continue
            seen_hashes[h] = n
            docs.append({**s, **d, "content_hash": h})
            if verbose: print(f"[ok ] {s['topic']:<20} {s['url'][:90]}")
        except Exception as e:
            failed.append({**s, "error": str(e)[:120]})
            if verbose: print(f"[ERR] {s['url'][:80]} -> {str(e)[:60]}")
        time.sleep(0.4)   # be polite

process(sources)
print(f"\nProcessed OK (unique): {len(docs)}   Failed: {len(failed)}   Duplicates skipped: {len(dup_log)}")

## 3. Auto-expand to reach 150+ unique sources (only runs if needed)
Follows on-topic links from pages already fetched (same domain; Wikipedia allowed in later rounds) for up to 4 rounds until the target is met. Every candidate goes through the same URL/content dedup.

In [ ]:
KEYWORDS = ["incoterm","customs","tariff","hs-code","harmonized","bill-of-lading","bill_of_lading","invoice","demurrage","detention",
            "letter-of-credit","letter_of_credit","trade-finance","freight","shipping","export","import","origin","container","cargo",
            "duty","clearance","port","logistic","forwarding","carrier","supply-chain","supply_chain","glossary","transport","trade","insurance","weather","surcharge","congestion","piracy","storm","typhoon","rates","risk"]

def expand(target=TARGET_SOURCES, per_page=8, rounds=4):
    for rd in range(1, rounds+1):
        if len(docs) >= target: break
        allow_wiki = rd >= 2
        cands = []
        for d in list(docs):
            dom = urlparse(d["url"]).netloc
            is_wiki = "wikipedia" in dom
            if is_wiki and not allow_wiki: continue
            n = 0
            for l in d.get("links", []):
                nl = normalize_url(l)
                if nl in seen_urls or urlparse(nl).netloc != normalize_url(d["url"]).split('/')[2]: continue
                if is_wiki and not ("/wiki/" in nl and ":" not in nl.split("/wiki/")[-1]): continue
                if any(k in nl.lower() for k in KEYWORDS) or nl.lower().endswith(".pdf"):
                    cands.append({"url": l, "norm_url": nl, "topic": d["topic"], "kind": "pdf" if l.lower().endswith(".pdf") else "html"})
                    seen_urls.add(nl); n += 1   # reserve so candidates are unique within the round
                    if n >= per_page: break
        print(f"Round {rd}: {len(cands)} candidates (have {len(docs)}/{target})")
        # un-reserve then let process() do the real dedup check
        for c in cands: seen_urls.discard(c["norm_url"])
        for c in cands:
            if len(docs) >= target + 10: break
            process([c])
        if not cands: break

if len(docs) < TARGET_SOURCES:
    print("Below target, expanding crawl...")
    expand()
print("Total unique processed sources:", len(docs))

## 3b. EXECUTION GATE — 150+ unique sources are REQUIRED
This cell **must run** and **must pass** before chunking starts. It counts the unique documents collected so far (URL-unique **and** content-unique). If fewer than 150, it automatically (1) fetches a reserve list of extra pages and (2) runs a wider link crawl. If it is still below 150 it **stops the notebook with an error** (so “Run all” cannot silently continue with too little data). Proof is saved to `rag_data/source_count_proof.json`.

In [ ]:
import datetime
WIKI_RESERVE = {
 "shipping_documents": ["Panamax","Suezmax","Capesize","Post-Panamax","Ultra_Large_Container_Vessel","Container_crane","Straddle_carrier","Reach_stacker","Gantry_crane","Pallet","Forklift","ISO_6346","Refrigerated_container","Tank_container","Flat_rack","Verified_Gross_Mass","Freight_forwarder","Ship_broker","Stevedore","Longshoreman","Tally_clerk","Shipping_company","Ship_chandler","Cargo_handling","Container_terminal","Ro-ro_ship","Car_carrier","Heavy-lift_ship","General_cargo_ship","Lighter_(barge)"],
 "customs": ["Free_port","Special_economic_zone","Export_processing_zone","Foreign-trade_zone","Tariff_war","Dumping_(pricing_policy)","Safeguard_(trade)","Trade_sanctions","Office_of_Foreign_Assets_Control","Export_Administration_Regulations","Dual-use_technology","Balance_of_trade","Customs_territory","Excise","Border_Gateway","Transit_(customs)","Customs_warehouse","Inward_processing","Outward_processing","Pre-shipment_inspection"],
 "demurrage_detention": ["Ship_registration","Flag_of_convenience","Port_state_control","Flag_state","Maritime_history","Shipping_container_corner_casting","Berth_(moorage)","Pilotage","Tugboat","Anchorage_(maritime)"],
 "trade_finance": ["Trade_credit","Accounts_receivable","Escrow","Exchange_rate","Current_account","Balance_of_payments","Credit_risk","Credit_default_swap","Invoice_discounting","Working_capital"],
 "ports": ["Port_of_Santos","Port_of_Valencia","Port_of_Felixstowe","Port_of_Tokyo","Port_of_Hong_Kong","Port_of_Qingdao","Port_of_Tianjin","Port_of_Guangzhou","Port_of_Piraeus","Port_Said","Tanjung_Pelepas","Colombo_Port","Port_Klang","Laem_Chabang_Port","Port_of_Durban"],
 "weather": ["Wind_wave_model","Significant_wave_height","Nor%27easter","Cyclone_Amphan","Gale","Maritime_weather_forecasting","Atlantic_hurricane_season","Pacific_typhoon_season","North_Indian_Ocean_cyclone_season"],
 "route_risks": ["Gulf_of_Aden","Red_Sea","Houthi%E2%80%93Israeli_conflict","Black_Sea_Grain_Initiative","Strait_of_Dover","Bosphorus","Dardanelles","Straits_of_Johor","Lombok_Strait","Sunda_Strait"],
 "marine_insurance": ["Hull_insurance","Salvage_(maritime_law)","Average_(maritime_law)","Insurance_broker","Insurance_policy","Risk_management","Actuary","Total_loss"],
}
GATE = {"required": TARGET_SOURCES, "stages": []}
def _unique_count():
    return len(docs), len({d["norm_url"] for d in docs}), len({d["content_hash"] for d in docs})

n, nu, nh = _unique_count()
GATE["stages"].append({"stage": "after_seed_and_expand", "documents": n})
print(f"[gate] unique documents so far: {n} (required {TARGET_SOURCES})")

if n < TARGET_SOURCES:
    print("[gate] below target -> fetching reserve sources...")
    reserve = [{"url": WIKI+t, "topic": tp, "kind": "html"} for tp, ts in WIKI_RESERVE.items() for t in ts]
    for i in range(0, len(reserve), 10):
        if len(docs) >= TARGET_SOURCES + 10: break
        process(reserve[i:i+10], verbose=False)
    n = len(docs); GATE["stages"].append({"stage": "after_reserve", "documents": n})
    print(f"[gate] after reserve: {n}")

if len(docs) < TARGET_SOURCES:
    print("[gate] still below target -> wide crawl (8 rounds)...")
    expand(target=TARGET_SOURCES, per_page=20, rounds=8)
    n = len(docs); GATE["stages"].append({"stage": "after_wide_crawl", "documents": n})
    print(f"[gate] after wide crawl: {n}")

n, nu, nh = _unique_count()
dom = Counter(urlparse(d["url"]).netloc.replace("www.", "") for d in docs)
proof = {"checked_at": datetime.datetime.now().isoformat(timespec="seconds"), "required": TARGET_SOURCES,
         "unique_documents": n, "unique_normalized_urls": nu, "unique_content_hashes": nh,
         "websites_html": sum(d["type"] == "html" for d in docs), "pdf_documents": sum(d["type"] == "pdf" for d in docs),
         "failed_sources": len(failed), "duplicates_skipped": len(dup_log), "distinct_domains": len(dom),
         "top_domains": dom.most_common(10), "stages": GATE["stages"], "gate_passed": bool(n >= TARGET_SOURCES and nu == n and nh == n)}
with open(f"{DATA_DIR}/source_count_proof.json", "w") as f: json.dump(proof, f, indent=2)
print(json.dumps({k: v for k, v in proof.items() if k != "top_domains"}, indent=1))
if not proof["gate_passed"]:
    raise RuntimeError(f"EXECUTION GATE FAILED: only {n} unique documents (need {TARGET_SOURCES}). "
                       f"{len(failed)} sources failed (see `failed` / rag_data after next save). Add reachable URLs to EXTRA_SOURCES or upload sources.csv, then run all cells again.")
print(f"\nGATE PASS: {n} unique documents >= {TARGET_SOURCES}  (URL-unique: {nu}, content-unique: {nh})")

## 4. Chunk (with chunk-level de-duplication)

In [ ]:
def chunk_text(text, size=CHUNK_CHARS, overlap=OVERLAP):
    paras = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size: cur += (" " if cur else "") + p
        else:
            if cur: chunks.append(cur)
            cur = (cur[-overlap:] + " " + p) if cur and len(p) < size else p
            while len(cur) > size * 1.5:      # very long paragraph
                chunks.append(cur[:size]); cur = cur[size-overlap:]
    if cur: chunks.append(cur)
    return [c for c in chunks if len(c) > 150]

chunks, seen_chunk_hashes, dropped_chunks = [], set(), 0
for d in docs:
    j = 0
    for c in chunk_text(d["text"]):
        ch = content_hash(c)
        if ch in seen_chunk_hashes:          # boilerplate / repeated text across sources
            dropped_chunks += 1; continue
        seen_chunk_hashes.add(ch)
        chunks.append({"chunk_id": hashlib.md5(f"{d['norm_url']}#{j}".encode()).hexdigest()[:16],
                       "source_url": d["url"], "title": d["title"], "topic": d["topic"],
                       "type": d["type"], "file": d["file"], "chunk_index": j, "text": c})
        j += 1
print(len(chunks), "unique chunks from", len(docs), "sources;", dropped_chunks, "duplicate chunks dropped")

## 5. Save to Google Drive (for the final app)
Outputs in `MyDrive/FreightQuote_AI/rag_data/`: `chunks.jsonl`, `sources.csv`, `failed_sources.csv`, `dedup_report.csv`, and `rag_kb.sqlite` (FTS5 full-text index, table `rag_chunks`).

In [ ]:
with open(f"{DATA_DIR}/chunks.jsonl","w",encoding="utf-8") as f:
    for c in chunks: f.write(json.dumps(c, ensure_ascii=False)+"\n")

with open(f"{DATA_DIR}/sources.csv","w",newline="",encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["url","normalized_url","topic","type","title","chars","chunks","content_sha256"])
    cc = Counter(c["source_url"] for c in chunks)
    for d in docs: w.writerow([d["url"], d["norm_url"], d["topic"], d["type"], d["title"], len(d["text"]), cc[d["url"]], d["content_hash"]])
with open(f"{DATA_DIR}/failed_sources.csv","w",newline="",encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["url","topic","error"])
    for x in failed: w.writerow([x["url"], x["topic"], x["error"]])
with open(f"{DATA_DIR}/dedup_report.csv","w",newline="",encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["duplicate_type","url","duplicate_of"])
    for row in dup_log: w.writerow(row)

db = f"{DATA_DIR}/rag_kb.sqlite"
if os.path.exists(db): os.remove(db)
con = sqlite3.connect(db)
con.execute("CREATE VIRTUAL TABLE rag_chunks USING fts5(chunk_id UNINDEXED, source_url UNINDEXED, title, topic UNINDEXED, text, tokenize='porter')")
con.executemany("INSERT INTO rag_chunks VALUES (?,?,?,?,?)", [(c["chunk_id"],c["source_url"],c["title"],c["topic"],c["text"]) for c in chunks])
con.commit(); con.close()
print("Saved to", DATA_DIR); print(os.listdir(DATA_DIR))
print("PDFs in rag_pdfs:", len([f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")]))
print("Duplicates logged:", Counter(r[0] for r in dup_log))

## 6. Build the FAISS vector index
Embeds every chunk with `all-MiniLM-L6-v2` (384-d, L2-normalized) and stores them in a `faiss.IndexFlatIP` (inner product = cosine similarity on normalized vectors). **Row *i* of the index = line *i* of `chunks.jsonl`.** Saved files: `faiss.index`, `embeddings.npy`, `faiss_meta.json`.

In [ ]:
import numpy as np, faiss, torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding on:", device, "| model:", EMB_MODEL)
embedder = SentenceTransformer(EMB_MODEL, device=device)

texts = [c["text"] for c in chunks]
emb = embedder.encode(texts, batch_size=64, show_progress_bar=True,
                      normalize_embeddings=True, convert_to_numpy=True).astype("float32")
dim = emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(emb)

faiss.write_index(index, f"{DATA_DIR}/faiss.index")
np.save(f"{DATA_DIR}/embeddings.npy", emb)
with open(f"{DATA_DIR}/faiss_meta.json","w") as f:
    json.dump({"model": EMB_MODEL, "dim": int(dim), "metric": "inner_product(normalized=cosine)",
               "index_type": "IndexFlatIP", "n_vectors": int(index.ntotal),
               "row_order": "same as chunks.jsonl line order"}, f, indent=2)
print(f"FAISS index saved: {index.ntotal} vectors x {dim} dims ->", f"{DATA_DIR}/faiss.index")

## 6b. Build the BM25 index (`rank-bm25`)
Builds a `BM25Okapi` index over every chunk (title + text, lower-cased, stop-words removed, light plural stemming). **Row *i* of the BM25 index = line *i* of `chunks.jsonl` = vector *i* of FAISS.**
Saved to `rag_data/`: `bm25_index.pkl` (the pickled `BM25Okapi`), `bm25_meta.json` (parameters + per-row `chunk_id`/`source_url`/`topic`/`title`) and `bm25_tokenizer.py` (the exact tokenizer, so the final app tokenizes queries identically).

In [ ]:
import pickle
from importlib.metadata import version as _pkg_version
from rank_bm25 import BM25Okapi

BM25_K1, BM25_B = 1.5, 0.75

TOKENIZER_SRC = """import re
BM25_STOP = set("a an the of to in on for and or is are was were be by with what how who why when does do did this that it its as at from vs versus which can will should".split())
def bm25_tokenize(text):
    out = []
    for t in re.findall(r"[a-z0-9]+", text.lower()):
        if len(t) < 2 or t in BM25_STOP: continue
        if len(t) > 4 and t.endswith("ies"): t = t[:-3] + "y"
        elif len(t) > 3 and t.endswith("s") and not t.endswith("ss"): t = t[:-1]
        out.append(t)
    return out
"""
exec(TOKENIZER_SRC)                       # defines bm25_tokenize + BM25_STOP in this notebook
with open(f"{DATA_DIR}/bm25_tokenizer.py", "w", encoding="utf-8") as f: f.write(TOKENIZER_SRC)

def build_bm25():
    """(Re)build BM25Okapi from the current `chunks`, save index + metadata, return the object."""
    tokenized = [bm25_tokenize(c["title"] + " " + c["text"]) for c in chunks]
    bm = BM25Okapi(tokenized, k1=BM25_K1, b=BM25_B)
    with open(f"{DATA_DIR}/bm25_index.pkl", "wb") as f: pickle.dump(bm, f, protocol=pickle.HIGHEST_PROTOCOL)
    meta = {"library": "rank-bm25", "library_version": _pkg_version("rank-bm25"), "class": "BM25Okapi",
            "k1": BM25_K1, "b": BM25_B, "n_docs": len(chunks), "avg_doc_len_tokens": round(float(bm.avgdl), 2),
            "tokenizer": "bm25_tokenizer.py :: bm25_tokenize (lowercase, [a-z0-9]+, stop-words removed, plural strip)",
            "indexed_text": "title + ' ' + chunk text",
            "row_order": "same as chunks.jsonl line order and FAISS row order",
            "rows": [{"chunk_id": c["chunk_id"], "source_url": c["source_url"], "topic": c["topic"], "title": c["title"]} for c in chunks]}
    with open(f"{DATA_DIR}/bm25_meta.json", "w", encoding="utf-8") as f: json.dump(meta, f, ensure_ascii=False)
    return bm

bm25_build = build_bm25()

# reload from disk to prove persistence (this loaded copy is what the retriever uses)
with open(f"{DATA_DIR}/bm25_index.pkl", "rb") as f: bm25 = pickle.load(f)
bm25_meta = json.load(open(f"{DATA_DIR}/bm25_meta.json", encoding="utf-8"))
assert bm25.corpus_size == len(chunks) == bm25_meta["n_docs"] == len(bm25_meta["rows"]), "BM25 / chunks mismatch"

def bm25_ids(q, k=20):
    toks = bm25_tokenize(q)
    if not toks: return []
    sc = bm25.get_scores(toks)
    order = np.argsort(-sc)[:k]
    return [chunks[i]["chunk_id"] for i in order if sc[i] > 0]

print(f"BM25 index saved: {bm25.corpus_size} chunks | k1={BM25_K1} b={BM25_B} | rank-bm25 {bm25_meta['library_version']}")
print("Files:", [f for f in os.listdir(DATA_DIR) if f.startswith("bm25")])
_top = bm25_ids("What is demurrage vs detention?", 3)
_by = {c["chunk_id"]: c for c in chunks}
for cid in _top: print(f"  - ({_by[cid]['topic']}) {_by[cid]['title'][:70]}")

## 6c. Self-check & auto-repair of weak topics
Runs the same per-topic test as the Verification cell (min. documents, min. chunks, key terms inside the topic). If any topic would show `LOW`, extra sources are fetched for **that topic only** (fallback seed pages first, then on-topic links from the topic's own pages), then chunks, FTS5, FAISS and BM25 are all rebuilt. Up to 3 attempts. If every topic is already OK this cell does nothing.

In [ ]:
import math, numpy as np, faiss
TOPIC_REQUIREMENTS = {
 "incoterms":           ["incoterms","fob","cif","exw","ddp"],
 "customs":             ["customs","duty","clearance"],
 "hs_codes":            ["harmonized","hs code","tariff"],
 "shipping_documents":  ["bill of lading","invoice","packing list"],
 "demurrage_detention": ["demurrage","detention","laytime"],
 "trade_finance":       ["letter of credit","trade finance"],
 "freight_rates":       ["freight rate","spot","contract","index"],
 "surcharges":          ["surcharge","bunker","fuel","peak season"],
 "ports":               ["port","terminal","container"],
 "congestion":          ["congestion","delays","backlog","supply chain"],
 "weather":             ["weather","storm","typhoon","cyclone","hurricane"],
 "route_risks":         ["piracy","red sea","suez","strait","security"],
 "marine_insurance":    ["marine insurance","cargo insurance","insurance","general average"],
}
MIN_DOCS_PER_TOPIC, MIN_CHUNKS_PER_TOPIC = 3, 15

FALLBACK_SEEDS = {   # extra pages tried first for a weak topic (failures are simply logged)
 "incoterms": ["United_Nations_Convention_on_Contracts_for_the_International_Sale_of_Goods","Sales_contract","Delivery_(commerce)","Bill_of_sale","Transfer_of_risk"],
 "customs": ["Customs_duty","Border_control","Tax_haven","Smuggling","Contraband","Trade_policy","International_trade"],
 "hs_codes": ["Nomenclature","Product_classification","Tariff_schedule","Commodity_classification","United_Nations_Statistics_Division","International_Standard_Industrial_Classification"],
 "shipping_documents": ["Waybill","Consignment_note","Bill_of_lading_(maritime)","Proof_of_delivery","Cargo_manifest","Shipping_label","Freight_bill"],
 "demurrage_detention": ["Lay_days","Laytime_and_demurrage","Dispatch_(shipping)","Charterparty","Maritime_commerce","Shipping_(economic)"],
 "trade_finance": ["Documentary_credit","Export_credit_insurance","Bill_of_exchange","Import_finance","Letters_of_credit","Commercial_bank","Short-term_financing"],
 "freight_rates": ["Shanghai_Containerized_Freight_Index","Harpex","Time_charter_equivalent","Charter_rate","Shipping_markets","Freight_futures","Container_freight_rate"],
 "surcharges": ["Bunker_surcharge","Sulphur_oxides","Marine_fuel","Heavy_fuel_oil","Low-sulphur_marine_fuels","Emissions_trading","Congestion_surcharge"],
 "ports": ["Port_of_New_York_and_New_Jersey","Port_of_Dubai","Port_of_Tanjung_Pelepas","Port_of_Colombo","Kandla_Port","Kolkata_Port","Ports_in_India","Container_terminal"],
 "congestion": ["Traffic_congestion","Bottleneck_(production)","Dwell_time","Supply_chain_disruption","Shipping_delays","Logistics_bottleneck","Queue_area"],
 "weather": ["Cyclone","Severe_weather","Tropical_cyclone_forecasting","Wind_wave","Sea_state","Marine_weather_forecasting","Maritime_meteorology","Typhoon_Haiyan"],
 "route_risks": ["Red_Sea","Gulf_of_Aden","Houthi_attacks_on_commercial_shipping","Somali_piracy","Strait_of_Hormuz_crisis","Chokepoint_(logistics)","Maritime_chokepoint","Shipping_lane"],
 "marine_insurance": ["Cargo_insurance","Hull_insurance","Insurance_policy","Marine_insurance_in_the_United_Kingdom","Average_(maritime_law)","Salvage_(maritime_law)","Insurance_broker"],
}

def compute_coverage():
    con_ = sqlite3.connect(f"{DATA_DIR}/rag_kb.sqlite")
    td = Counter(d["topic"] for d in docs); tc = Counter(c["topic"] for c in chunks)
    rep_, weak_ = {}, []
    for topic, kws in TOPIC_REQUIREMENTS.items():
        hits = [k for k in kws if con_.execute("SELECT count(*) FROM rag_chunks WHERE rag_chunks MATCH ? AND topic = ?", (f'"{k}"', topic)).fetchone()[0] > 0]
        need = max(2, math.ceil(0.6*len(kws)))
        ok = len(hits) >= need and td.get(topic,0) >= MIN_DOCS_PER_TOPIC and tc.get(topic,0) >= MIN_CHUNKS_PER_TOPIC
        rep_[topic] = {"docs": td.get(topic,0), "chunks": tc.get(topic,0), "hits": hits, "need": need, "total": len(kws), "ok": ok}
        if not ok: weak_.append(topic)
    con_.close()
    return rep_, weak_

def topup_topic(topic, extra_docs=6, rounds=3):
    """Fetch more sources for ONE topic: fallback seeds first, then ranked on-topic links from that topic's own pages."""
    before = len(docs)
    process([{"url": WIKI+t, "topic": topic, "kind": "html"} for t in FALLBACK_SEEDS.get(topic, [])], verbose=False)
    slugs = [k.replace(" ", "_") for k in TOPIC_REQUIREMENTS[topic]] + [k.replace(" ", "-") for k in TOPIC_REQUIREMENTS[topic]]
    for _ in range(rounds):
        if len(docs) - before >= extra_docs: break
        cands = []
        for d in [x for x in docs if x["topic"] == topic]:
            dom = urlparse(d["url"]).netloc; is_wiki = "wikipedia" in dom
            for l in d.get("links", []):
                nl = normalize_url(l)
                if nl in seen_urls or urlparse(nl).netloc != normalize_url(d["url"]).split("/")[2]: continue
                if is_wiki and not ("/wiki/" in nl and ":" not in nl.split("/wiki/")[-1]): continue
                cands.append({"url": l, "norm_url": nl, "topic": topic, "kind": "pdf" if l.lower().endswith(".pdf") else "html",
                              "_score": sum(sl in nl.lower() for sl in slugs)})
        cands.sort(key=lambda c: -c["_score"])
        seen_c, pick = set(), []
        for c in cands:
            if c["norm_url"] in seen_c: continue
            seen_c.add(c["norm_url"]); pick.append(c)
            if len(pick) >= 12: break
        if not pick: break
        process(pick, verbose=False)
    return len(docs) - before

def rebuild_everything():
    """Re-chunk and rebuild chunks.jsonl, CSVs, FTS5, FAISS and BM25 from the current `docs`."""
    global chunks, seen_chunk_hashes, dropped_chunks, emb, index
    chunks, seen_chunk_hashes, dropped_chunks = [], set(), 0
    for d in docs:
        j = 0
        for c in chunk_text(d["text"]):
            ch = content_hash(c)
            if ch in seen_chunk_hashes: dropped_chunks += 1; continue
            seen_chunk_hashes.add(ch)
            chunks.append({"chunk_id": hashlib.md5(f"{d['norm_url']}#{j}".encode()).hexdigest()[:16], "source_url": d["url"], "title": d["title"],
                           "topic": d["topic"], "type": d["type"], "file": d["file"], "chunk_index": j, "text": c}); j += 1
    with open(f"{DATA_DIR}/chunks.jsonl", "w", encoding="utf-8") as f:
        for c in chunks: f.write(json.dumps(c, ensure_ascii=False) + "\n")
    cc = Counter(c["source_url"] for c in chunks)
    with open(f"{DATA_DIR}/sources.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["url","normalized_url","topic","type","title","chars","chunks","content_sha256"])
        for d in docs: w.writerow([d["url"], d["norm_url"], d["topic"], d["type"], d["title"], len(d["text"]), cc[d["url"]], d["content_hash"]])
    with open(f"{DATA_DIR}/failed_sources.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["url","topic","error"])
        for x in failed: w.writerow([x["url"], x["topic"], x["error"]])
    with open(f"{DATA_DIR}/dedup_report.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["duplicate_type","url","duplicate_of"])
        for row in dup_log: w.writerow(row)
    db_ = f"{DATA_DIR}/rag_kb.sqlite"
    if os.path.exists(db_): os.remove(db_)
    c_ = sqlite3.connect(db_)
    c_.execute("CREATE VIRTUAL TABLE rag_chunks USING fts5(chunk_id UNINDEXED, source_url UNINDEXED, title, topic UNINDEXED, text, tokenize='porter')")
    c_.executemany("INSERT INTO rag_chunks VALUES (?,?,?,?,?)", [(c["chunk_id"],c["source_url"],c["title"],c["topic"],c["text"]) for c in chunks])
    c_.commit(); c_.close()
    emb = embedder.encode([c["text"] for c in chunks], batch_size=64, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
    faiss.write_index(index, f"{DATA_DIR}/faiss.index"); np.save(f"{DATA_DIR}/embeddings.npy", emb)
    with open(f"{DATA_DIR}/faiss_meta.json", "w") as f:
        json.dump({"model": EMB_MODEL, "dim": int(emb.shape[1]), "metric": "inner_product(normalized=cosine)", "index_type": "IndexFlatIP",
                   "n_vectors": int(index.ntotal), "row_order": "same as chunks.jsonl line order"}, f, indent=2)
    global bm25, bm25_meta
    build_bm25()
    with open(f"{DATA_DIR}/bm25_index.pkl", "rb") as f: bm25 = pickle.load(f)
    bm25_meta = json.load(open(f"{DATA_DIR}/bm25_meta.json", encoding="utf-8"))
    print(f"  rebuilt: {len(docs)} docs, {len(chunks)} chunks, FAISS {index.ntotal}, BM25 {bm25.corpus_size}")

_, weak = compute_coverage()
print("Weak topics before repair:", weak or "none")
attempt = 0
while weak and attempt < 3:
    attempt += 1
    print(f"-- auto-repair attempt {attempt}: topping up {weak}")
    for t in weak: print(f"   {t}: +{topup_topic(t)} new documents")
    rebuild_everything()
    _, weak = compute_coverage()
    print("   weak topics now:", weak or "none")
if len(docs) < TARGET_SOURCES:
    print(f"Only {len(docs)} documents (< {TARGET_SOURCES}); expanding crawl again...")
    expand(); rebuild_everything()
print("Self-check done. Documents:", len(docs), "| weak topics:", weak or "none")

## 7. Verification (this is the evidence for the checklist)

In [ ]:
import math
# ---- per-topic coverage requirements (v3: all 13 topics, checked INSIDE each topic) ----
TOPIC_REQUIREMENTS = {
 "incoterms":           ["incoterms","fob","cif","exw","ddp"],
 "customs":             ["customs","duty","clearance"],
 "hs_codes":            ["harmonized","hs code","tariff"],
 "shipping_documents":  ["bill of lading","invoice","packing list"],
 "demurrage_detention": ["demurrage","detention","laytime"],
 "trade_finance":       ["letter of credit","trade finance"],
 "freight_rates":       ["freight rate","spot","contract","index"],
 "surcharges":          ["surcharge","bunker","fuel","peak season"],
 "ports":               ["port","terminal","container"],
 "congestion":          ["congestion","delays","backlog","supply chain"],
 "weather":             ["weather","storm","typhoon","cyclone","hurricane"],
 "route_risks":         ["piracy","red sea","suez","strait","security"],
 "marine_insurance":    ["marine insurance","cargo insurance","insurance","general average"],
}
MIN_DOCS_PER_TOPIC, MIN_CHUNKS_PER_TOPIC, MIN_PDFS = 3, 15, 3

con = sqlite3.connect(f"{DATA_DIR}/rag_kb.sqlite")
n_chunks = con.execute("SELECT count(*) FROM rag_chunks").fetchone()[0]
src_with_chunks = {c["source_url"] for c in chunks}
n_src = len(docs)
n_norm_unique = len({d["norm_url"] for d in docs})
n_hash_unique = len({d["content_hash"] for d in docs})
n_pdf = len([d for d in docs if d["type"]=="pdf"]); n_html = len(docs)-n_pdf
topic_docs   = Counter(d["topic"] for d in docs)
topic_chunks = Counter(c["topic"] for c in chunks)

idx2 = faiss.read_index(f"{DATA_DIR}/faiss.index")      # reload from disk to prove persistence
faiss_ok = (idx2.ntotal == n_chunks == len(chunks))
with open(f"{DATA_DIR}/bm25_index.pkl", "rb") as _f: bm25_chk = pickle.load(_f)       # reload from disk to prove persistence
_bm = json.load(open(f"{DATA_DIR}/bm25_meta.json", encoding="utf-8"))
bm25_ok = (bm25_chk.corpus_size == n_chunks == len(_bm["rows"]) and [r["chunk_id"] for r in _bm["rows"]] == [c["chunk_id"] for c in chunks])

print(f"Unique documents                 : {n_src}  (target {TARGET_SOURCES})   [websites {n_html} / PDFs {n_pdf}]")
print(f"Unique normalized URLs           : {n_norm_unique}")
print(f"Unique content hashes            : {n_hash_unique}")
print(f"Duplicates removed               : {dict(Counter(r[0] for r in dup_log))} + {dropped_chunks} duplicate chunks")
print(f"Chunks in KB (FTS5)              : {n_chunks}")
print(f"Vectors in FAISS index           : {idx2.ntotal}  (matches chunks: {faiss_ok})")
print(f"Docs in BM25 index (BM25Okapi)   : {bm25_chk.corpus_size}  (matches chunks + metadata: {bm25_ok})")
print()
print(f"{'topic':<22}{'docs':>5}{'chunks':>8}   key terms found (inside topic)")
coverage_report, weak_topics = {}, []
for topic, kws in TOPIC_REQUIREMENTS.items():
    hits = [k for k in kws if con.execute("SELECT count(*) FROM rag_chunks WHERE rag_chunks MATCH ? AND topic = ?", (f'"{k}"', topic)).fetchone()[0] > 0]
    need = max(2, math.ceil(0.6*len(kws)))
    nd, nc = topic_docs.get(topic,0), topic_chunks.get(topic,0)
    ok = len(hits) >= need and nd >= MIN_DOCS_PER_TOPIC and nc >= MIN_CHUNKS_PER_TOPIC
    coverage_report[topic] = {"documents": nd, "chunks": nc, "terms_found": hits, "terms_required": need, "terms_total": len(kws), "ok": ok}
    if not ok: weak_topics.append(topic)
    print(f"{'['+('OK ' if ok else 'LOW')+'] '+topic:<22}{nd:>5}{nc:>8}   {len(hits)}/{len(kws)} (need {need}): {', '.join(hits)}")

checks = {
  ">=150 unique documents (URL + content unique)": n_src >= TARGET_SOURCES and n_norm_unique == n_src and n_hash_unique == n_src,
  "no duplicate chunks": len({content_hash(c['text']) for c in chunks}) == len(chunks),
  "FAISS matches chunks": faiss_ok,
  "BM25 index + metadata match chunks": bm25_ok,
  "all 13 topics covered": not weak_topics,
}
print()
for k,v in checks.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
if n_pdf < MIN_PDFS: print(f"  WARN  only {n_pdf} PDF(s) processed (recommended >= {MIN_PDFS}); add PDF links to EXTRA_SOURCES if you need more")
print("\nRESULT:", "PASS" if all(checks.values()) else "NEEDS MORE SOURCES -> add to EXTRA_SOURCES / sources.csv and re-run",
      "| weak topics:", weak_topics or "none")

In [ ]:
def search(q, k=3):
    q = " OR ".join(re.findall(r"\w+", q))
    rows = con.execute("SELECT title, topic, source_url, snippet(rag_chunks,4,'[',']','…',24) FROM rag_chunks WHERE rag_chunks MATCH ? ORDER BY rank LIMIT ?", (q,k)).fetchall()
    for r in rows: print(f"- ({r[1]}) {r[0]}\n  {r[3]}\n  {r[2]}\n")

def vsearch(q, k=3):
    qv = embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    D, I = idx2.search(qv, k)
    for score, i in zip(D[0], I[0]):
        c = chunks[i]; print(f"- score={score:.3f} ({c['topic']}) {c['title']}\n  {c['text'][:200]}...\n  {c['source_url']}\n")

# quick smoke test of both retrievers (the full 12-question RAG test with citations is in the next section)
print("-- keyword (FTS5) --"); search("What is demurrage vs detention?", 2)
print("-- keyword (BM25) --")
for _cid in bm25_ids("What is demurrage vs detention?", 2):
    _c = {c["chunk_id"]: c for c in chunks}[_cid]; print(f"- ({_c['topic']}) {_c['title']}\n  {_c['text'][:200]}...\n  {_c['source_url']}\n")
print("-- semantic (FAISS) --"); vsearch("What is demurrage vs detention?", 2)

## 8. RAG answers with numbered citations (hybrid retrieval)
**Retrieval:** FAISS semantic search + FTS5 keyword search + **BM25 (rank-bm25)** fused with Reciprocal Rank Fusion, max 2 chunks per source.
**Answer:** the most relevant *sentences* from the retrieved chunks, each followed by a citation number `[n]`; numbers are assigned in order of first appearance and listed in a **Sources** block.
**Citation verification (`verify_citations`)** proves, for every answer, that (a) every `[n]` has a source entry, (b) every listed source is cited, (c) every cited sentence appears verbatim in the chunk it cites, (d) the chunk belongs to the cited URL, and (e) that URL is a document in the knowledge base.

In [ ]:
STOP = set("a an the of to in on for and or is are was were be by with what how who why when does do did this that it its as at from vs versus which can will should".split())
kb_by_id = {c["chunk_id"]: c for c in chunks}
kb_urls  = {d["url"] for d in docs}

def fts_ids(q, k=20):
    terms = [t for t in re.findall(r"\w+", q.lower()) if t not in STOP and len(t) > 2]
    if not terms: return []
    rows = con.execute("SELECT chunk_id FROM rag_chunks WHERE rag_chunks MATCH ? ORDER BY rank LIMIT ?", (" OR ".join(terms), k)).fetchall()
    return [r[0] for r in rows]

def retrieve(q, k=6, per_source=2, pool=30):
    qv = embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    D, I = idx2.search(qv, pool)
    vec_rank = {chunks[i]["chunk_id"]: r for r, i in enumerate(I[0]) if i >= 0}
    fts_rank = {cid: r for r, cid in enumerate(fts_ids(q))}
    bm_rank  = {cid: r for r, cid in enumerate(bm25_ids(q, pool))}
    fused = {cid: (1/(60+vec_rank[cid]) if cid in vec_rank else 0) + (1/(60+fts_rank[cid]) if cid in fts_rank else 0)
                  + (1/(60+bm_rank[cid]) if cid in bm_rank else 0)
             for cid in set(vec_rank) | set(fts_rank) | set(bm_rank)}
    out, per = [], Counter()
    for cid, sc in sorted(fused.items(), key=lambda x: -x[1]):
        src = kb_by_id[cid]["source_url"]
        if per[src] >= per_source: continue
        per[src] += 1; out.append(cid)
        if len(out) >= k: break
    return qv, out

def split_sents(t):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", t) if 50 <= len(s.strip()) <= 350]

def answer_with_citations(q, n_sent=4, max_per_source=2, min_sim=0.25, k=6):
    qv, hit_ids = retrieve(q, k=k)
    evidence = " ".join(kb_by_id[c]["text"] for c in hit_ids)
    cand = [(s, cid) for cid in hit_ids for s in split_sents(kb_by_id[cid]["text"])]
    base = {"question": q, "evidence_text": evidence, "evidence_chunks": hit_ids}
    if not cand:
        return {**base, "answer": "No supporting evidence found in the knowledge base.", "sources": [], "sentences": []}
    sv = embedder.encode([s for s, _ in cand], batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    sims = sv @ qv[0]
    picked, per, seen_txt = [], Counter(), set()
    for j in np.argsort(-sims):
        s, cid = cand[j]; src = kb_by_id[cid]["source_url"]; key = norm_text(s)
        if sims[j] < min_sim or per[src] >= max_per_source or key in seen_txt: continue
        picked.append({"text": s, "chunk_id": cid, "sim": round(float(sims[j]), 3)}); per[src] += 1; seen_txt.add(key)
        if len(picked) >= n_sent: break
    if not picked:
        return {**base, "answer": "No sufficiently relevant evidence found in the knowledge base.", "sources": [], "sentences": []}
    num, sources_out = {}, []
    for p in picked:                                    # number sources by first appearance
        c = kb_by_id[p["chunk_id"]]
        if c["source_url"] not in num:
            num[c["source_url"]] = len(num) + 1
            sources_out.append({"n": num[c["source_url"]], "url": c["source_url"], "title": c["title"], "topic": c["topic"]})
        p["n"] = num[c["source_url"]]
    answer = " ".join(f"{p['text']} [{p['n']}]" for p in picked)
    return {**base, "answer": answer, "sources": sources_out, "sentences": picked}

def verify_citations(res):
    problems = []
    ans = res["answer"]
    cited = {int(x) for x in re.findall(r"\[(\d+)\]", ans)}
    src_by_n = {s["n"]: s for s in res["sources"]}
    if not res["sentences"]: problems.append("no cited sentences")
    for n in cited:
        if n not in src_by_n: problems.append(f"[{n}] has no source entry")
    for n in src_by_n:
        if n not in cited: problems.append(f"source [{n}] listed but never cited")
        if src_by_n[n]["url"] not in kb_urls: problems.append(f"source [{n}] is not a KB document")
    for st in res["sentences"]:
        chunk = kb_by_id.get(st["chunk_id"])
        if chunk is None: problems.append("unknown chunk"); continue
        if norm_text(st["text"]) not in norm_text(chunk["text"]): problems.append(f"sentence for [{st['n']}] not found verbatim in its chunk")
        if src_by_n.get(st["n"], {}).get("url") != chunk["source_url"]: problems.append(f"[{st['n']}] points to a different URL than its chunk")
        if f"{st['text']} [{st['n']}]" not in ans: problems.append(f"[{st['n']}] marker missing from answer text")
    return (not problems), problems
print("RAG functions ready")

In [ ]:
# 12 test questions (>= 10 required) - at least one per topic. Each has expected evidence keywords (any one must appear in retrieved text).
QUESTIONS = [
 ("What is the difference between demurrage and detention?",                        ["demurrage","detention"]),
 ("Under FOB versus CIF, who pays for freight and insurance?",                       ["fob","cif"]),
 ("What is a letter of credit and how does it protect an exporter?",                 ["letter of credit"]),
 ("How are goods classified using HS codes (the Harmonized System)?",                ["harmonized","hs code"]),
 ("Which documents are needed for an international shipment, such as the bill of lading and commercial invoice?", ["bill of lading","invoice","packing list"]),
 ("How are container freight rates set, and which indices track them?",              ["freight rate","index","spot"]),
 ("What surcharges do carriers add, for example bunker, fuel or peak season charges?", ["surcharge","bunker","fuel"]),
 ("What causes port congestion and how does it delay cargo?",                        ["congestion","delay"]),
 ("How do typhoons, hurricanes and storms affect shipping schedules and routes?",    ["storm","typhoon","hurricane","cyclone","weather"]),
 ("What route risks such as Red Sea attacks, piracy or canal closures affect shipping?", ["red sea","piracy","suez","strait"]),
 ("What does marine cargo insurance cover, and what is general average?",            ["insurance","general average"]),
 ("What does a customs broker do and how does customs clearance work?",              ["customs broker","broker","clearance"]),
]
rag_results = []
for i, (q, kws) in enumerate(QUESTIONS, 1):
    for attempt, (ns, ms, kk) in enumerate([(4, 0.25, 6), (5, 0.18, 10), (6, 0.10, 14)], 1):   # retry wider if a question fails
        res = answer_with_citations(q, n_sent=ns, min_sim=ms, k=kk)
        cit_ok, probs = verify_citations(res)
        ev = norm_text(res["evidence_text"])
        kw_hit = [k for k in kws if k in ev]
        if cit_ok and res["sentences"] and kw_hit: break
    row = {"n": i, "question": q, "answer": res["answer"], "sources": res["sources"],
           "num_citations": len(res["sentences"]), "distinct_sources": len(res["sources"]),
           "citations_valid": cit_ok, "problems": probs, "keywords_expected": kws, "keywords_found": kw_hit,
           "attempts": attempt, "pass": bool(cit_ok and res["sentences"] and kw_hit)}
    rag_results.append(row)
    print(f"\nQ{i}. {q}\n{res['answer']}\nSources:")
    for s in res["sources"]: print(f"  [{s['n']}] ({s['topic']}) {s['title'][:70]} - {s['url']}")
    print(f"  -> citations valid: {cit_ok} {probs or ''} | evidence keywords found: {kw_hit or 'NONE'} | attempt {attempt} | {'PASS' if row['pass'] else 'FAIL'}")

n_q = len(rag_results); n_pass = sum(r["pass"] for r in rag_results)
rag_summary = {
  "questions_run": n_q, "questions_passed": n_pass,
  "all_citations_valid": all(r["citations_valid"] for r in rag_results),
  "avg_citations_per_answer": round(sum(r["num_citations"] for r in rag_results)/max(1,n_q), 2),
  "avg_distinct_sources_per_answer": round(sum(r["distinct_sources"] for r in rag_results)/max(1,n_q), 2),
  "failed_questions": [r["n"] for r in rag_results if not r["pass"]],
  "all_questions_passed": n_pass == n_q,
}
rag_summary["result"] = "PASS" if (n_q >= 10 and rag_summary["all_citations_valid"] and n_pass >= 0.9*n_q) else "FAIL"
with open(f"{DATA_DIR}/rag_test_results.json", "w", encoding="utf-8") as f: json.dump(rag_results, f, ensure_ascii=False, indent=1)
with open(f"{DATA_DIR}/rag_answers.md", "w", encoding="utf-8") as f:
    f.write("# RAG test answers with numbered citations\n\n")
    for r in rag_results:
        f.write(f"## Q{r['n']}. {r['question']}\n\n{r['answer']}\n\n**Sources**\n")
        for s in r["sources"]: f.write(f"- [{s['n']}] {s['title']} - {s['url']}\n")
        f.write(f"\n_citations valid: {r['citations_valid']} | result: {'PASS' if r['pass'] else 'FAIL'}_\n\n")
print("\n" + "="*60); print("RAG TEST SUMMARY:", json.dumps(rag_summary, indent=1)); print("RAG TEST RESULT:", rag_summary["result"])

## 9. Record statistics
Writes `kb_statistics.json` and `kb_statistics.md` to `rag_data/` (unique documents, website/PDF counts, duplicates removed, chunks, per-topic counts, FTS5 + FAISS index stats, coverage and RAG-test results). Screenshot the printed table for the checklist.

In [ ]:
import datetime
lens = [len(c["text"]) for c in chunks]
mb = lambda p: round(os.path.getsize(p)/1e6, 2)
all_topics = sorted(set(topic_docs) | set(topic_chunks) | set(TOPIC_REQUIREMENTS))
STATS = {
 "generated_at": datetime.datetime.now().isoformat(timespec="seconds"),
 "documents": {
   "unique_documents": len(docs), "websites_html": n_html, "pdf_documents": n_pdf,
   "pdf_files_in_rag_pdfs": len([f for f in os.listdir(PDF_DIR) if f.lower().endswith(".pdf")]),
   "failed_sources": len(failed), "target": TARGET_SOURCES, "target_met": len(docs) >= TARGET_SOURCES,
   "total_characters": int(sum(len(d["text"]) for d in docs)),
 },
 "deduplication": {
   "duplicates_removed_by_type": dict(Counter(r[0] for r in dup_log)),
   "total_duplicate_urls_or_docs_removed": len(dup_log),
   "duplicate_chunks_dropped": dropped_chunks,
   "unique_normalized_urls": n_norm_unique, "unique_content_hashes": n_hash_unique,
 },
 "chunks": {
   "total_chunks": len(chunks), "chunk_chars_setting": CHUNK_CHARS, "overlap_setting": OVERLAP,
   "avg_chunk_chars": round(sum(lens)/max(1,len(lens)), 1), "min_chunk_chars": min(lens), "max_chunk_chars": max(lens),
 },
 "topics": {t: {"documents": topic_docs.get(t, 0), "chunks": topic_chunks.get(t, 0)} for t in all_topics},
 "topic_coverage": coverage_report,
 "indexes": {
   "fts5": {"file": "rag_kb.sqlite", "table": "rag_chunks", "rows": n_chunks, "size_mb": mb(f"{DATA_DIR}/rag_kb.sqlite"), "tokenizer": "porter"},
   "faiss": {"file": "faiss.index", "type": "IndexFlatIP (cosine on normalized vectors)", "model": EMB_MODEL,
             "dimension": int(idx2.d), "vectors": int(idx2.ntotal), "size_mb": mb(f"{DATA_DIR}/faiss.index"),
             "embeddings_npy_size_mb": mb(f"{DATA_DIR}/embeddings.npy")},
 },
 "rag_test": rag_summary,
}
STATS["indexes"]["bm25"] = {"file": "bm25_index.pkl", "metadata": "bm25_meta.json", "class": "BM25Okapi", "library": "rank-bm25 " + bm25_meta["library_version"],
                            "k1": BM25_K1, "b": BM25_B, "documents": int(bm25.corpus_size), "size_mb": mb(f"{DATA_DIR}/bm25_index.pkl")}
with open(f"{DATA_DIR}/kb_statistics.json", "w", encoding="utf-8") as f: json.dump(STATS, f, indent=2, ensure_ascii=False)

d_, dd, ch, ix = STATS["documents"], STATS["deduplication"], STATS["chunks"], STATS["indexes"]
md_lines = [
 "# FreightQuote AI - Knowledge Base Statistics", f"_Generated {STATS['generated_at']}_", "",
 "| Metric | Value |", "|---|---|",
 f"| Unique documents | {d_['unique_documents']} (target {d_['target']}: {'met' if d_['target_met'] else 'NOT met'}) |",
 f"| Websites (HTML) | {d_['websites_html']} |", f"| PDF documents | {d_['pdf_documents']} (files in rag_pdfs: {d_['pdf_files_in_rag_pdfs']}) |",
 f"| Failed sources | {d_['failed_sources']} |", f"| Total characters | {d_['total_characters']:,} |",
 f"| Duplicates removed (by type) | {dd['duplicates_removed_by_type'] or 'none'} |",
 f"| Duplicate chunks dropped | {dd['duplicate_chunks_dropped']} |",
 f"| Total chunks | {ch['total_chunks']} (avg {ch['avg_chunk_chars']} chars, min {ch['min_chunk_chars']}, max {ch['max_chunk_chars']}) |",
 f"| FTS5 index | {ix['fts5']['rows']} rows, {ix['fts5']['size_mb']} MB |",
 f"| FAISS index | {ix['faiss']['vectors']} vectors x {ix['faiss']['dimension']} dims, {ix['faiss']['type']}, {ix['faiss']['size_mb']} MB |",
 f"| BM25 index | {ix['bm25']['documents']} chunks, {ix['bm25']['class']} (k1={ix['bm25']['k1']}, b={ix['bm25']['b']}), {ix['bm25']['size_mb']} MB |",
 f"| Embedding model | {EMB_MODEL} |",
 f"| RAG test | {rag_summary['questions_passed']}/{rag_summary['questions_run']} passed, citations valid: {rag_summary['all_citations_valid']} -> {rag_summary['result']} |",
 "", "## Per-topic coverage", "", "| Topic | Documents | Chunks | Key terms | OK |", "|---|---|---|---|---|",
]
for t in all_topics:
    cr = coverage_report.get(t, {})
    md_lines.append(f"| {t} | {topic_docs.get(t,0)} | {topic_chunks.get(t,0)} | {len(cr.get('terms_found',[]))}/{cr.get('terms_total','-')} | {'yes' if cr.get('ok') else 'NO'} |")
with open(f"{DATA_DIR}/kb_statistics.md", "w", encoding="utf-8") as f: f.write("\n".join(md_lines))
print("\n".join(md_lines)); print("\nSaved: kb_statistics.json, kb_statistics.md ->", DATA_DIR)

## 9b. Final checklist (your 4 requirements)
Prints PASS / FAIL for each requirement and saves `final_checklist.json`. If something shows FAIL, the message says what to do.

In [ ]:
_rag_all   = rag_summary["questions_run"] >= 12 and rag_summary["all_questions_passed"]
_bm25_ok   = bm25.corpus_size == len(chunks) and os.path.exists(f"{DATA_DIR}/bm25_index.pkl") and os.path.exists(f"{DATA_DIR}/bm25_meta.json")
_weak_now  = [t for t, r in coverage_report.items() if not r["ok"]]
CHECKLIST = {
 "1. rank-bm25 installed, BM25Okapi built from chunks, index + metadata saved": _bm25_ok,
 "2a. 150+ unique documents collected (URL + content unique)":                  len(docs) >= TARGET_SOURCES and n_norm_unique == len(docs) and n_hash_unique == len(docs),
 "2b. FAISS index created and matches chunks":                                  faiss_ok,
 "3.  all 13 topics PASS (none LOW)":                                           len(coverage_report) == 13 and not _weak_now,
 "4a. all 12 RAG questions pass":                                               _rag_all,
 "4b. all citations valid":                                                     rag_summary["all_citations_valid"],
}
print(f"Unique documents: {len(docs)} | chunks: {len(chunks)} | FAISS: {idx2.ntotal} | BM25: {bm25.corpus_size}")
print(f"RAG questions passed: {rag_summary['questions_passed']}/{rag_summary['questions_run']} | failed: {rag_summary['failed_questions'] or 'none'}\n")
for k, v in CHECKLIST.items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
if _weak_now: print("\nLOW topics:", _weak_now, "-> add sources for them to EXTRA_SOURCES / FALLBACK_SEEDS and run all cells again.")
if rag_summary["failed_questions"]: print("Failing questions:", rag_summary["failed_questions"], "-> add on-topic sources and run all cells again.")
print("\nFINAL RESULT:", "ALL PASS" if all(CHECKLIST.values()) else "SOME CHECKS FAILED")
with open(f"{DATA_DIR}/final_checklist.json", "w") as f: json.dump({"checks": CHECKLIST, "all_pass": all(CHECKLIST.values())}, f, indent=2)
assert all(CHECKLIST.values()), "FINAL CHECKLIST FAILED - see the FAIL lines above"


## 10. Hand-off to the final application (Member 4)
```python
import json, faiss, numpy as np, re
from sentence_transformers import SentenceTransformer
D = '/content/drive/MyDrive/FreightQuote_AI/rag_data'
KB    = [json.loads(l) for l in open(f'{D}/chunks.jsonl', encoding='utf-8')]   # row i == FAISS vector i
index = faiss.read_index(f'{D}/faiss.index')
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
q = model.encode(['who pays freight under CIF?'], normalize_embeddings=True).astype('float32')
scores, ids = index.search(q, 5)
hits = [KB[i] for i in ids[0]]          # cite as [1], [2]... using hit['source_url'] / hit['title']
# BM25 alternative (same row order as KB):
#   import pickle, sys; sys.path.append(D); from bm25_tokenizer import bm25_tokenize
#   bm25 = pickle.load(open(f'{D}/bm25_index.pkl','rb'))            # needs: pip install rank-bm25
#   top = bm25.get_scores(bm25_tokenize('who pays freight under CIF?')).argsort()[::-1][:5]; hits = [KB[i] for i in top]
# keyword search alternative: {D}/rag_kb.sqlite  (FTS5 table: rag_chunks)
# statistics: {D}/kb_statistics.json | test answers: {D}/rag_answers.md, rag_test_results.json
```
For the app's own answers, copy `answer_with_citations` / `verify_citations` from section 8 (they only need `chunks`, the FAISS index, the FTS5 connection, the BM25 index and the embedder).

**Checklist evidence to screenshot:** the BM25 cell output, Verification cell (150+ unique documents, 13 topics OK, FAISS count, `PASS`), the RAG test summary (`PASS`, 12 questions, valid citations), the Final checklist cell, `kb_statistics.md`, the `rag_pdfs/` Drive folder, and `dedup_report.csv`.

If a topic is weak or the document count is short: add sources to `EXTRA_SOURCES` (or upload `sources.csv`) and re-run.